# phase_2 LLM branch — finetune the **flat** scene-graph extractor (Qwen2.5-3B QLoRA)

Step 6: 4-bit QLoRA finetune of `Qwen/Qwen2.5-3B-Instruct` to turn **report text + the list of
available regions** into the **flat per-region findings JSON** (`sg_schema.py`):

```json
{ "left lower lung zone": [ {"finding": "atelectasis", "presence": "yes", "progression": "worsened"} ] }
```

A small model only has to copy a finding *name* (not a pipe-delimited relation string); a
deterministic mapper (`compact_from_flat`) turns the JSON back into ImaGenome relation/cue
strings at assembly time. Drive-resumable via an **OAuth user token** (not a service account).

**Run BEFORE this notebook (local, no GPU):** build the SFT data and upload it as a Kaggle dataset
```
python build_sft_dataset.py --metadata data/mimic_metadata_final.jsonl \
    --scene-root <chest-imagenome dir> --out phase_2/_work/sg_sft
```


## CONFIG — edit these

In [ ]:
import os
# ===== code comes from GitHub (cloned next cell) =====
PHASE2_SRC = "/kaggle/working/repo/phase_2"
# ===== SFT data: build_sft_dataset.py output uploaded as a Kaggle dataset (train.jsonl/val.jsonl) =====
SG_SFT  = "/kaggle/input/sg-sft"

# ===== training =====
MODEL    = "Qwen/Qwen2.5-3B-Instruct"   # 3B is enough for closed-vocab extraction after SFT;
                                        # set "Qwen/Qwen2.5-7B-Instruct" to compare via eval cell
EPOCHS   = 2
RUN_NAME = "sg_lora"
RESUME   = 0          # 1 from the 2nd session on (pulls checkpoints from Drive, continues)

# ===== durable checkpoints (Google Drive via YOUR OAuth token — see next section) =====
DRIVE_FOLDER_ID = "1c6AJ3fjsL449kiMK4xiXfnzwzA4gIo0O"   # CHEX-DATA folder id
RCLONE_REMOTE   = "dhint:sg_lora_runs"                  # = CHEX-DATA/sg_lora_runs

for k, v in dict(PHASE2_SRC=PHASE2_SRC, SG_SFT=SG_SFT, MODEL=MODEL, EPOCHS=EPOCHS,
                 RUN_NAME=RUN_NAME, RESUME=RESUME, DRIVE_FOLDER_ID=DRIVE_FOLDER_ID,
                 RCLONE_REMOTE=RCLONE_REMOTE).items():
    os.environ[k] = str(v)
print("config | MODEL =", MODEL, "| EPOCHS =", EPOCHS, "| RESUME =", RESUME)


In [ ]:
# --- get the code from GitHub. Internet ON. ---
!rm -rf /kaggle/working/repo && git clone -q https://github.com/hiennguyendang/phase_2_3_4_5 /kaggle/working/repo
!ls /kaggle/working/repo/phase_2 | head


## 1 — install rclone + auth via YOUR Google account (OAuth token)

Service accounts have **no Drive storage quota** and 403 on upload to a *My Drive* folder — so we
authenticate as **you** with an OAuth token (files upload to your own quota).

**One-time setup, on a machine WITH a browser:**
1. Install rclone (https://rclone.org/downloads/ or `winget install Rclone.Rclone`).
2. `rclone authorize "drive"` → log in with the account that **owns CHEX-DATA** → *Allow*.
3. Copy the whole `{...}` token JSON it prints.
4. Kaggle **Add-ons → Secrets → Add**: label **`GDRIVE_TOKEN`**, value = that `{...}`.
5. Keep the CHEX-DATA folder id in `DRIVE_FOLDER_ID` (CONFIG). The cell below runs a real write-test.

In [ ]:
%%bash
set -e
if ! command -v rclone >/dev/null 2>&1; then
  cd /kaggle/working && curl -sLO https://downloads.rclone.org/rclone-current-linux-amd64.zip
  unzip -q -o rclone-current-linux-amd64.zip && cp rclone-*-linux-amd64/rclone /usr/local/bin/ && chmod +x /usr/local/bin/rclone
fi
rclone version | head -1


In [ ]:
import os
# Drive remote 'dhint' via YOUR OAuth token (NOT a service account: SA has no storage quota and
# 403s on upload to My Drive). Token = `rclone authorize "drive"` -> Kaggle Secret GDRIVE_TOKEN.
# Graceful: if missing/unwritable, training still runs WITHOUT sync.
SYNC_OK = "0"
try:
    from kaggle_secrets import UserSecretsClient
    sec = UserSecretsClient()
    token = sec.get_secret("GDRIVE_TOKEN").strip()
    os.environ["RCLONE_CONFIG_DHINT_TYPE"] = "drive"
    os.environ["RCLONE_CONFIG_DHINT_TOKEN"] = token
    os.environ["RCLONE_CONFIG_DHINT_SCOPE"] = "drive"
    os.environ["RCLONE_CONFIG_DHINT_ROOT_FOLDER_ID"] = os.environ["DRIVE_FOLDER_ID"]
    os.environ.pop("RCLONE_CONFIG_DHINT_SERVICE_ACCOUNT_FILE", None)  # drop stale SA from a prior run
    # OPTIONAL own Google API client -> private query quota (avoids shared-client RATE_LIMIT on long runs)
    for k_sec, k_env in [("GDRIVE_CLIENT_ID", "RCLONE_CONFIG_DHINT_CLIENT_ID"),
                         ("GDRIVE_CLIENT_SECRET", "RCLONE_CONFIG_DHINT_CLIENT_SECRET")]:
        try:
            os.environ[k_env] = sec.get_secret(k_sec).strip()
        except Exception:
            pass
    using_own = "own client" if os.environ.get("RCLONE_CONFIG_DHINT_CLIENT_ID") else "rclone shared client"
    remote = os.environ["RCLONE_REMOTE"]
    if os.system('rclone mkdir "%s"' % remote) == 0 and \
       os.system('echo ok | rclone rcat "%s/_write_test.txt"' % remote) == 0:
        SYNC_OK = "1"
        os.system('rclone delete "%s/_write_test.txt" 2>/dev/null' % remote)
        print(f"remote OK (OAuth, write verified, {using_own}) ->", remote)
    else:
        print("[WARN] Drive write FAILED -> check GDRIVE_TOKEN (own account, write scope) + DRIVE_FOLDER_ID")
except Exception as e:
    print("[WARN] GDRIVE_TOKEN secret not set -> NO Drive sync:", e)
os.environ["SYNC_OK"] = SYNC_OK
print("SYNC_OK =", SYNC_OK)


## 2 — copy code + install LLM deps
If the trl API errors (it shifts across versions), pin `trl==0.11.4 transformers==4.44.2`.

In [ ]:
!rm -rf /kaggle/working/phase_2 && cp -r "$PHASE2_SRC" /kaggle/working/phase_2
%cd /kaggle/working/phase_2
!pip install -q "transformers>=4.44" "trl>=0.11,<0.12" "peft>=0.12" "bitsandbytes>=0.43" "accelerate>=0.33" "datasets>=2.20"
!python -c "import json; n=sum(1 for _ in open('$SG_SFT/train.jsonl')); print('train samples:', n)"  # sanity


## 3 — finetune (Drive-synced, auto multi-GPU)
Pushes the run dir to Drive every checkpoint. If the session dies, set `RESUME=1` in CONFIG and
Run All again (pulls checkpoints from Drive, continues).

**2×T4:** the cell auto-detects GPU count and launches **DDP via `torchrun`** when >1 (each GPU a
full model copy, batch split → ~2× faster). 1 GPU → plain `python`. Per-device `--batch 4`; if you
OOM on T4 16GB drop it to 2. (`device_map="auto"` alone would NOT speed up — it just shards one
model across GPUs; the script switches to per-rank placement under DDP.)

In [ ]:
import os, torch
NPROC = torch.cuda.device_count()
REMOTE = "%s/%s" % (os.environ["RCLONE_REMOTE"], os.environ["RUN_NAME"])
S = ('--sync-remote "%s"' % REMOTE) if os.environ.get("SYNC_OK") == "1" else ""
R = "--resume" if int(os.environ["RESUME"]) else ""
# DDP can't safely pull-resume inside the script (ranks would race on the dir) -> pull once here
if int(os.environ["RESUME"]) and NPROC > 1 and os.environ.get("SYNC_OK") == "1":
    os.system('rclone copy "%s" /kaggle/working/sg_lora --quiet' % REMOTE)
os.environ.update(S_FLAG=S, R_FLAG=R)
launcher = ("torchrun --nproc_per_node=%d" % NPROC) if NPROC > 1 else "python"
print("GPUs:", NPROC, "| launcher:", launcher, "| flags:", S or "(no sync)", R or "(fresh)")
!{launcher} finetune_sg_llm.py --data-dir "$SG_SFT" --out /kaggle/working/sg_lora \
  --model "$MODEL" --epochs $EPOCHS --batch 4 $S_FLAG $R_FLAG


## 4 — evaluate (per-finding accuracy)
Scores the adapter on `val.jsonl`: format validity, presence P/R/F1 (with vs without region =
localization gap), 3-class progression accuracy, and a per-finding table. Run this with
`MODEL=...-7B-Instruct` (re-finetune) to check whether 3B leaves any F1 on the table.

In [ ]:
!python eval_sg_llm.py --val "$SG_SFT/val.jsonl" --lora /kaggle/working/sg_lora \
  --model "$MODEL" --out /kaggle/working/sg_eval.json --limit 2000
import os
if os.environ.get("SYNC_OK") == "1":
    os.system('rclone copy /kaggle/working/sg_eval.json "%s/%s" --quiet' % (os.environ["RCLONE_REMOTE"], os.environ["RUN_NAME"]))


## 5 — grab the LoRA adapters
Already on Drive (final push). Pull anywhere: `rclone copy dhint:sg_lora_runs/sg_lora ./sg_lora`.
Step 7 (`build_pseudo_scene_graph.py`) loads these on the base model.

In [ ]:
import os
if os.environ.get("SYNC_OK") == "1":
    os.system('rclone copy /kaggle/working/sg_lora "%s/%s" --quiet' % (os.environ["RCLONE_REMOTE"], os.environ["RUN_NAME"]))
    print("adapters pushed -> %s/%s" % (os.environ["RCLONE_REMOTE"], os.environ["RUN_NAME"]))
!ls -lh /kaggle/working/sg_lora | head
